# Generate 2023 UK Data Sythetically

### Vehicle Registrations Trends UK 2024 vs 2023

https://www.acea.auto/pc-registrations/new-car-registrations-0-8-in-2024-battery-electric-13-6-market-share/

In [ ]:
import pandas as pd

In [156]:
id_columns = ['oem', 'year_report', 'month_report', 'level_0_country',
       'level_1_region_name', 'level_2_district_postcode',
       'level_2_district_town_name', 
       'model_class', 'model_class_specific', 'body_type', 'fuel_type',
       'doors', 'Energy_Source']

def explode_id_to_columns(df: pd.DataFrame, id_columns: list) -> pd.DataFrame:

    # extract columns from vehicle_count_id
    for i, col in enumerate(id_columns):
        df[col] = df['vehicle_count_id'].str.split('_', expand=True)[i]
        
    df['year_report'] = df['year_report'].astype(int)
    df['month_report'] = df['month_report'].astype(int)
    df['doors'] = df['doors'].astype(int)

    return df

In [146]:
total_count_2023 = [131994, 74441, 287825, 132990, 145204, 177266, 86657, 272610, 153529, 156525, 141092]
total_count_2024 = [147876, 84886, 317768, 134274, 147678, 179263, 84575, 275239, 144288, 153610, 140786]

# calculate monthly percentage change
monthly_percentage_change = [round((old - new) / new * 100, 4) for old, new in zip(total_count_2023, total_count_2024)]
trend_adjustment_map = dict(zip(range(1, 12), monthly_percentage_change))
trend_adjustment_map

{1: -10.7401,
 2: -12.3047,
 3: -9.4229,
 4: -0.9563,
 5: -1.6753,
 6: -1.114,
 7: 2.4617,
 8: -0.9552,
 9: 6.4046,
 10: 1.8977,
 11: 0.2174}

In [147]:
import pandas as pd
df = pd.read_parquet("../data/fact_registered_vehicles.parquet")

In [157]:
df['vehicle_count_id'] = df[id_columns].apply(lambda x: '_'.join(x.astype(str)), axis=1)
df_grouped = df.groupby(['vehicle_count_id']).agg({
    'vehicle_count': 'sum'
}).reset_index()

In [158]:
df = explode_id_to_columns(df_grouped, id_columns)

In [161]:
assert df[df['vehicle_count_id'].duplicated(keep=False)].sort_values('vehicle_count_id').shape[0] == 0, "Duplicate vehicle_count_id found in the DataFrame"

## Generate 2023 Data using Trend Adjustment Map

In [162]:
df_2023 = df[['vehicle_count_id','month_report', 'vehicle_count']].copy()
df_2023['vehicle_count_id'] = df_2023['vehicle_count_id'].str.replace('_2024_', '_2023_')
df_2023['year_report'] = 2023

# Apply Trend Adjustment Map if and only if vehicle count > 1
import numpy as np
df_2023['vehicle_count'] = np.where(df_2023['vehicle_count'] > 1, np.floor(df_2023['vehicle_count'] * (1 + df_2023['month_report'].map(trend_adjustment_map).fillna(0) / 100)), df_2023['vehicle_count'])
df_2023['vehicle_count'] = df_2023['vehicle_count'].astype(int)

In [163]:
df_2023.head()

,vehicle_count_id,month_report,vehicle_count,year_report
0,ASTON MARTIN_2023_10_England_Bedford Borough_M...,10,1,2023
1,ASTON MARTIN_2023_10_England_Bedford Borough_M...,10,1,2023
2,ASTON MARTIN_2023_10_England_Bedford Borough_M...,10,1,2023
3,ASTON MARTIN_2023_10_England_Bracknell Forest_...,10,1,2023
4,ASTON MARTIN_2023_10_England_Bradford_LS29_Ilk...,10,1,2023


In [167]:
assert df_2023.shape[0] == df.shape[0], "Data frame shapes do not match"

In [168]:
monthly_totals_2024 = df.groupby(['month_report']).agg({'vehicle_count': 'sum'}).reset_index().set_index('month_report').rename(columns={'vehicle_count': 'total_vehicle_count_2024'})
monthly_totals_2023 = df_2023.groupby(['month_report']).agg({'vehicle_count': 'sum'}).reset_index().set_index('month_report').rename(columns={'vehicle_count': 'total_vehicle_count_2023'})

# calculate monthly percentage change 2024 to 2023
monthly_percentage_change = ((monthly_totals_2023['total_vehicle_count_2023'] - monthly_totals_2024['total_vehicle_count_2024']) / monthly_totals_2024['total_vehicle_count_2024'] * 100).fillna(0)
monthly_percentage_change

month_report
1    -10.160618
2    -10.105646
3    -10.567240
4     -7.343572
5     -7.911301
6     -8.327258
7      0.265696
8     -7.549394
9      1.358734
10     0.106921
11     0.000000
12     0.000000
dtype: float64

In [169]:
trend_adjustment_map

{1: -10.7401,
 2: -12.3047,
 3: -9.4229,
 4: -0.9563,
 5: -1.6753,
 6: -1.114,
 7: 2.4617,
 8: -0.9552,
 9: 6.4046,
 10: 1.8977,
 11: 0.2174}

In [170]:
df[df['vehicle_count_id'] == 'AUDI_2024_3_England_Leicester_LE1_Leicester_Q2_TFSI S LINE_SUV_PETROL_5_NORMAL']

,vehicle_count_id,vehicle_count,oem,year_report,month_report,level_0_country,level_1_region_name,level_2_district_postcode,level_2_district_town_name,model_class,model_class_specific,body_type,fuel_type,doors,Energy_Source
28973,AUDI_2024_3_England_Leicester_LE1_Leicester_Q2...,42,AUDI,2024,3,England,Leicester,LE1,Leicester,Q2,TFSI S LINE,SUV,PETROL,5,NORMAL


In [171]:
df_2023[df_2023['vehicle_count_id'] == 'AUDI_2023_3_England_Leicester_LE1_Leicester_Q2_TFSI S LINE_SUV_PETROL_5_NORMAL']

,vehicle_count_id,month_report,vehicle_count,year_report
28973,AUDI_2023_3_England_Leicester_LE1_Leicester_Q2...,3,38,2023


In [172]:
df_2023 = explode_id_to_columns(df_2023, id_columns)

In [173]:
assert set(df_2023.shape) == set(df.shape), "Data frame shapes do not match"

In [174]:
df_new = pd.concat([df, df_2023], ignore_index=True).reset_index(drop=True)

assert df_new[df_new['vehicle_count_id'].duplicated(keep=False)].sort_values('vehicle_count_id').shape[0] == 0, "Duplicate vehicle_count_id found in the DataFrame"

In [175]:
assert set(df_new['year_report'].unique().tolist())==set([2023, 2024]), "Expected years in the data 2023 and 2024 are not found"

In [180]:
df_new.shape

(625476, 15)

In [181]:
df_new.head()

,vehicle_count_id,vehicle_count,oem,year_report,month_report,level_0_country,level_1_region_name,level_2_district_postcode,level_2_district_town_name,model_class,model_class_specific,body_type,fuel_type,doors,Energy_Source
0,ASTON MARTIN_2024_10_England_Bedford Borough_M...,1,ASTON MARTIN,2024,10,England,Bedford Borough,MK43,Kempston Rural,DBX,V8,SUV,PETROL,5,NORMAL
1,ASTON MARTIN_2024_10_England_Bedford Borough_M...,1,ASTON MARTIN,2024,10,England,Bedford Borough,MK44,Wilden,DBX,V8,SUV,PETROL,5,NORMAL
2,ASTON MARTIN_2024_10_England_Bedford Borough_M...,1,ASTON MARTIN,2024,10,England,Bedford Borough,MK44,Wilden,VANTAGE,V8,SPORTSCOUPE,PETROL,3,NORMAL
3,ASTON MARTIN_2024_10_England_Bracknell Forest_...,1,ASTON MARTIN,2024,10,England,Bracknell Forest,RG42,Warfield,DBX,V8,SUV,PETROL,5,NORMAL
4,ASTON MARTIN_2024_10_England_Bradford_LS29_Ilk...,1,ASTON MARTIN,2024,10,England,Bradford,LS29,Ilkley,VANTAGE,V8,SPORTSCOUPE,PETROL,3,NORMAL


In [177]:
df_new.to_parquet("../data/fact_registered_vehicles_2023_2024.parquet", index=False)

In [178]:
_df = pd.read_parquet("../data/fact_registered_vehicles_2023_2024.parquet")
_df['year_report'].unique()

array([2024, 2023])

In [179]:
_df.shape

(625476, 15)

# Validate SQL Database

In [143]:
import sqlite3
import pandas as pd

# Connect to the traditional star schema SQLite database
conn = sqlite3.connect("../registered_vehicles.sqlite")

# validate available years in sqlite
df_years = pd.read_sql_query("SELECT DISTINCT year_report FROM DimTime", conn)
print("Available years in SQLite:")
print(df_years)

Available years in SQLite:
   year_report
0         2024


In [142]:

# Query to get top OEMs by total vehicle registrations with market share calculation
sql_query = """
WITH country_totals AS (
    -- Calculate total vehicles per country per time period
    SELECT 
        f.geography_country_key,
        f.time_key,
        gc.country_name,
        SUM(f.vehicle_count) as total_country_vehicles
    FROM FactRegisteredVehicles f
    JOIN DimGeographyCountry gc ON f.geography_country_key = gc.geography_country_key
    GROUP BY f.geography_country_key, f.time_key, gc.country_name
),
oem_country_totals AS (
    -- Calculate OEM vehicles per country per time period
    SELECT 
        f.oem_key,
        f.geography_country_key,
        f.time_key,
        o.oem_name,
        o.oem_category,
        gc.country_name,
        SUM(f.vehicle_count) as oem_country_vehicles
    FROM FactRegisteredVehicles f
    JOIN DimOEM o ON f.oem_key = o.oem_key
    JOIN DimGeographyCountry gc ON f.geography_country_key = gc.geography_country_key
    GROUP BY f.oem_key, f.geography_country_key, f.time_key, o.oem_name, o.oem_category, gc.country_name
)
SELECT 
    oct.oem_name,
    oct.oem_category,
    oct.country_name,
    SUM(oct.oem_country_vehicles) as total_vehicles,
    SUM(ct.total_country_vehicles) as country_total_vehicles,
    ROUND(
        (SUM(oct.oem_country_vehicles) * 100.0 / SUM(ct.total_country_vehicles)), 2
    ) as market_share_percent,
    COUNT(DISTINCT oct.time_key) as months_active
FROM oem_country_totals oct
JOIN country_totals ct ON oct.geography_country_key = ct.geography_country_key 
    AND oct.time_key = ct.time_key
GROUP BY oct.oem_name, oct.oem_category, oct.country_name
ORDER BY oct.country_name, total_vehicles DESC;
"""

# Execute query and get results
df_market_share = pd.read_sql_query(sql_query, conn)

# Close the connection
conn.close()

print("=== TOP OEMs BY COUNTRY WITH MARKET SHARE ===")
print(f"Total records: {len(df_market_share)}")
print()

# Display results by country
for country in df_market_share['country_name'].unique():
    country_data = df_market_share[df_market_share['country_name'] == country].head(10)
    print(f"📍 {country.upper()}:")
    print("-" * 60)
    for _, row in country_data.iterrows():
        print(f"  {row['oem_name']} ({row['oem_category']})")
        print(f"    Vehicles: {row['total_vehicles']:,}")
        print(f"    Market Share: {row['market_share_percent']}%")
        print(f"    Active Months: {row['months_active']}")
        print()
    print("=" * 60)
    print()

# Show overall top performers
print("🏆 OVERALL TOP 10 OEMs (All Countries Combined):")
print("-" * 60)
overall_top = df_market_share.groupby(['oem_name', 'oem_category']).agg({
    'total_vehicles': 'sum',
    'months_active': 'sum'
}).reset_index().sort_values('total_vehicles', ascending=False).head(10)

for _, row in overall_top.iterrows():
    print(f"  {row['oem_name']} ({row['oem_category']})")
    print(f"    Total Vehicles: {row['total_vehicles']:,}")
    print(f"    Total Active Months: {row['months_active']}")
    print()

=== TOP OEMs BY COUNTRY WITH MARKET SHARE ===
Total records: 91

📍 ENGLAND:
------------------------------------------------------------
  BMW (Luxury)
    Vehicles: 110,912
    Market Share: 23.56%
    Active Months: 12

  AUDI (Luxury)
    Vehicles: 108,117
    Market Share: 22.97%
    Active Months: 12

  MERCEDES-BENZ (Luxury)
    Vehicles: 93,013
    Market Share: 19.76%
    Active Months: 12

  VOLVO (Premium)
    Vehicles: 58,668
    Market Share: 12.46%
    Active Months: 12

  MINI (Mass Market)
    Vehicles: 41,582
    Market Share: 8.83%
    Active Months: 12

  PORSCHE (Luxury)
    Vehicles: 17,593
    Market Share: 3.74%
    Active Months: 12

  LEXUS (Premium)
    Vehicles: 14,775
    Market Share: 3.14%
    Active Months: 12

  JAGUAR (Luxury)
    Vehicles: 14,666
    Market Share: 3.12%
    Active Months: 12

  POLESTAR (Mass Market)
    Vehicles: 8,454
    Market Share: 1.8%
    Active Months: 12

  ASTON MARTIN (Luxury)
    Vehicles: 910
    Market Share: 0.19%
    Ac

In [182]:
# Verify both years are in the combined data file
import pandas as pd
df_combined = pd.read_parquet("../data/fact_registered_vehicles_2023_2024.parquet")
print("Shape of combined data:", df_combined.shape)
print("Years in combined data:", sorted(df_combined['year_report'].unique()))
print("Record counts by year:")
year_counts = df_combined['year_report'].value_counts().sort_index()
print(year_counts)

Shape of combined data: (625476, 15)
Years in combined data: [np.int64(2023), np.int64(2024)]
Record counts by year:
year_report
2023    312738
2024    312738
Name: count, dtype: int64


In [185]:
# Fix DimTime table by adding 2023 entries
import pandas as pd

# Load existing DimTime (2024 only)
dim_time_2024 = pd.read_parquet("../data/star_schema/DimTime.parquet")
print("Original DimTime shape:", dim_time_2024.shape)
print("Original years:", sorted(dim_time_2024['year_report'].unique()))
print("Data types:")
print(dim_time_2024.dtypes)
print("\\nSample data:")
print(dim_time_2024.head(3))

# Create 2023 entries by copying 2024 structure
dim_time_2023 = dim_time_2024.copy()
dim_time_2023['year_report'] = 2023
dim_time_2023['time_key'] = dim_time_2023['time_key'] - 100  # 202401 becomes 202301

# Handle year_month - convert to string first if it's datetime
if dim_time_2023['year_month'].dtype == 'datetime64[ns]':
    dim_time_2023['year_month'] = pd.to_datetime(dim_time_2023['year_month'].astype(str).str.replace('2024', '2023'))
else:
    dim_time_2023['year_month'] = dim_time_2023['year_month'].astype(str).str.replace('2024', '2023')

# quarter stays the same
dim_time_2023['quarter'] = dim_time_2023['quarter']

# Handle year_quarter
dim_time_2023['year_quarter'] = dim_time_2023['year_quarter'].astype(str).str.replace('2024', '2023')

# Combine 2023 and 2024 data
dim_time_combined = pd.concat([dim_time_2023, dim_time_2024], ignore_index=True)
dim_time_combined = dim_time_combined.sort_values('time_key').reset_index(drop=True)

print("\\nCombined DimTime shape:", dim_time_combined.shape)
print("Combined years:", sorted(dim_time_combined['year_report'].unique()))
print("\\nSample of combined DimTime:")
print(dim_time_combined.head(8))

# Save the updated DimTime table
dim_time_combined.to_parquet("../data/star_schema/DimTime.parquet", index=False)
print("\\n✅ Updated DimTime.parquet saved with both 2023 and 2024 data")

Original DimTime shape: (24, 6)
Original years: [np.int64(2023), np.int64(2024)]
Data types:
time_key                 int64
year_report              int64
month_report             int64
year_month      datetime64[ns]
quarter                  int64
year_quarter            object
dtype: object

Sample data:
   time_key  year_report  month_report year_month  quarter year_quarter
0    202301         2023             1 2023-01-01        1      2023-Q1
1    202302         2023             2 2023-02-01        1      2023-Q1
2    202303         2023             3 2023-03-01        1      2023-Q1

Combined DimTime shape: (48, 7)
Combined years: [np.int64(2023), np.int64(2024)]

New column added:
year_month_date type: datetime64[ns]

Sample of combined DimTime with new column:
   time_key  year_report  month_report year_month year_month_date
0    202201         2023             1 2023-01-01      2023-01-01
1    202202         2023             2 2023-02-01      2023-02-01
2    202203         2023